<a href="https://colab.research.google.com/github/belleasi/Machine-Learning-models/blob/main/unsupervised/pca_kmeans_wine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Set random seed
np.random.seed(42)

print("=" * 60)
print("UNSUPERVISED LEARNING: PCA + K-MEANS")
print("=" * 60)

# Load dataset
wine = load_wine()
X = wine.data
feature_names = wine.feature_names

df = pd.DataFrame(X, columns=feature_names)

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nSummary statistics:")
print(df.describe())

# Standardize data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Use PCA to reduce dimensions to 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("\nExplained variance ratio:")
print(pca.explained_variance_ratio_)

# Find best K using silhouette score
k_values = range(2, 8)
silhouette_scores = []
inertia_values = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)

    silhouette_scores.append(silhouette_score(X_scaled, labels))
    inertia_values.append(model.inertia_)

    print(f"K={k}: Inertia={model.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.3f}")

# Train final KMeans model
best_k = 3
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

print("\nFinal Model:")
print("Number of clusters:", best_k)
print("Final inertia:", kmeans.inertia_)
print("Final silhouette score:", silhouette_score(X_scaled, clusters))

# Add cluster labels to dataframe
df["Cluster"] = clusters

print("\nCluster counts:")
print(df["Cluster"].value_counts())

# Visualization 1: PCA scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, alpha=0.7)
plt.title("K-Means Clustering on Wine Dataset using PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(label="Cluster")
plt.show()

# Visualization 2: Elbow method
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertia_values, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.grid(True)
plt.show()

# Visualization 3: Silhouette score
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), silhouette_scores, marker="o")
plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()